# 01 — EDA: raw FD001 training data audit

**Goal:** check the raw training file *before* any cleaning, so later steps (RUL labels, engine-ID splits, rolling features) rest on verified assumptions.

Checks: shape, column names, engine counts, cycle order, missing values, duplicate rows.

Data: `train_FD001.txt`, `test_FD001.txt` and `RUL_FD001.txt` in `data/raw/`. Run this notebook from the `notebooks/` folder.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from rul_prediction.data import audit_raw_data, load_cmapss_file

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train_FD001.txt"
train = load_cmapss_file(TRAIN_PATH)

## Shape and column names

26 columns = engine ID, cycle, 3 operating settings, 21 sensors.

In [2]:
print("shape:", train.shape)
print(list(train.columns))
train.head()

shape: (20631, 26)
['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [3]:
train.dtypes.value_counts()

float64    22
int64       4
Name: count, dtype: int64

## Structural audit summary

In [4]:
summary = audit_raw_data(train)
for check, value in summary.items():
    print(f"{check:36s} {value}")

n_rows                               20631
n_columns                            26
n_engines                            100
engine_id_min                        1
engine_id_max                        100
cycles_per_engine_min                128
cycles_per_engine_median             199.0
cycles_per_engine_max                362
missing_values_total                 0
duplicate_rows                       0
duplicate_engine_cycle_pairs         0
cycles_start_at_1_and_consecutive    True
sorted_by_engine_then_cycle          True


## Cycles per engine

Each training engine runs to failure, so its last cycle is its failure point. Trajectory length varies by engine, which is why RUL must be computed per engine.

In [5]:
cycles_per_engine = train.groupby("engine_id")["cycle"].max()
cycles_per_engine.describe()

count    100.000000
mean     206.310000
std       46.342749
min      128.000000
25%      177.000000
50%      199.000000
75%      229.250000
max      362.000000
Name: cycle, dtype: float64

## Missing values per column

In [6]:
missing = train.isna().sum()
missing[missing > 0] if missing.any() else "No missing values in any column"

'No missing values in any column'

## Test file audit

Test engines stop **before** failure, so trajectories are shorter than in training. The same structural checks must still pass, because predictions are made at each test engine's last cycle.

In [7]:
from rul_prediction.data import load_rul_file

test = load_cmapss_file(PROJECT_ROOT / "data" / "raw" / "test_FD001.txt")
test_summary = audit_raw_data(test)
for check, value in test_summary.items():
    print(f"{check:36s} {value}")

n_rows                               13096
n_columns                            26
n_engines                            100
engine_id_min                        1
engine_id_max                        100
cycles_per_engine_min                31
cycles_per_engine_median             133.5
cycles_per_engine_max                303
missing_values_total                 0
duplicate_rows                       0
duplicate_engine_cycle_pairs         0
cycles_start_at_1_and_consecutive    True
sorted_by_engine_then_cycle          True


## Test RUL answer key alignment

`RUL_FD001.txt` has one value per line and no engine column; line *i* is test engine *i*. Its engine IDs must match the test file exactly, otherwise predictions would be compared with the wrong answers.

The RUL **values** are the official test labels: they are used only for final evaluation, so we deliberately do not summarise their distribution here (it must not influence an RUL cap or hyperparameters).

In [8]:
test_rul = load_rul_file(PROJECT_ROOT / "data" / "raw" / "RUL_FD001.txt")
test_engine_ids = sorted(test["engine_id"].unique())

# Fail loudly: a mismatch means predictions would be scored against the wrong engines.
assert test_rul["engine_id"].tolist() == test_engine_ids, "RUL file does not match test engine IDs"
print("RUL rows:", len(test_rul), "| test engines:", len(test_engine_ids), "| engine IDs match")

RUL rows: 100 | test engines: 100 | engine IDs match


## Findings

_All values below come from the outputs of the cells above._

- **Rows / columns:** 20,631 rows × 26 columns (engine ID, cycle, 3 settings, 21 sensors). `engine_id` and `cycle` are integers; the other 24 columns are floats.
- **Engines:** 100 engines, IDs 1–100.
- **Trajectory length (cycles per engine):** min 128, median 199, mean 206.3, max 362. Engines fail at very different cycles, so RUL must be computed per engine.
- **Missing values:** none in any column.
- **Duplicates:** 0 duplicate rows and 0 duplicate (engine, cycle) pairs.
- **Cycle order:** every engine's cycles start at 1 and run consecutively with no gaps; rows are already sorted by engine, then cycle.

**Test file and answer key:**

- **Test file:** 13,096 rows × 26 columns, 100 engines (IDs 1–100); 31 to 303 cycles per engine (median 133.5). No missing values or duplicates; cycles start at 1, are consecutive and sorted. Test trajectories are shorter than training ones because they stop before failure.
- **`RUL_FD001.txt`:** 100 values, one per test engine; engine IDs match the test file exactly. Values are kept for final evaluation only and are not summarised here.


**Implication:** all structural checks pass, so no structural cleaning is needed. RUL labels and engine-ID splits can be built directly on this table. Sensor content (e.g. constant sensors) is checked in a later step, and nothing is removed here.